# HW6 - Q1 - 4 bit QAT VGG-16 MAC training

vgg16_quant (4bit) trained on 16 x 16 weight stationary systolic array

In [1]:
import os, time
import torch
import torch.nn as nn
import torchvision
import torchvision.transforms as transforms

from models.quant_layer import act_quantization, QuantConv2d as QuantConv2d4
#from models.resnet_quant import resnet20_quant
#from models.resnet_quant_2bit import resnet20_quant_2bit
from models.vgg_quant import VGG16_quant



device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
batch_size = 128
normalize = transforms.Normalize(mean=[0.491, 0.482, 0.447], std=[0.247, 0.243, 0.262])
trainset = torchvision.datasets.CIFAR10(root='./data', train=True, download=True, transform=transforms.Compose(
    [transforms.RandomCrop(32, padding=4), transforms.RandomHorizontalFlip(), transforms.ToTensor(), normalize]))
testset = torchvision.datasets.CIFAR10(root='./data', train=False, download=True, transform=transforms.Compose(
    [transforms.ToTensor(), normalize]))
trainloader = torch.utils.data.DataLoader(trainset, batch_size=batch_size, shuffle=True, num_workers=2)
testloader = torch.utils.data.DataLoader(testset, batch_size=batch_size, shuffle=False, num_workers=2)


def evaluate(model):
    model.eval(); correct = 0
    with torch.no_grad():
        for x, y in testloader:
            correct += model(x.to(device)).argmax(1).eq(y.to(device)).sum().item()
    return 100. * correct / len(testset)


def train_qat(model, name, epochs, lr=0.04, wd=1e-4, retrain=False):
    """QAT with SGD + cosine LR. Skips training if a checkpoint exists. Per-epoch logs are not printed."""
    os.makedirs('result/' + name, exist_ok=True)
    path = 'result/%s/model_best.pth.tar' % name
    if os.path.exists(path) and not retrain:
        model.load_state_dict(torch.load(path)['state_dict']); return model
    model.to(device)
    opt = torch.optim.SGD(model.parameters(), lr=lr, momentum=0.9, weight_decay=wd)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, epochs)
    crit = nn.CrossEntropyLoss(); best = 0
    for ep in range(epochs):
        model.train()
        for x, y in trainloader:
            x, y = x.to(device), y.to(device)
            loss = crit(model(x), y)
            opt.zero_grad(); loss.backward(); opt.step()
        sched.step()
        acc = evaluate(model)
        if acc > best:
            best = acc
            torch.save({'epoch': ep + 1, 'state_dict': model.state_dict(), 'best_prec': best}, path)
    model.load_state_dict(torch.load(path)['state_dict'])
    return model

In [2]:
model_vgg4 = VGG16_quant().to(device)
print(model_vgg4.features[0]) #float Conv2d (first layer is not quantized) 
print(model_vgg4.features[3]) # QuantConv2d (4bit)

model_vgg4 = train_qat(model_vgg4, 'vgg16_quant_4bit', epochs=80) 
print('VGG16 4-bit test accuracy: %.2f%%' % evaluate(model_vgg4))


QuantConv2d(
  3, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
  (weight_quant): weight_quantize_fn()
)
QuantConv2d(
  64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
  (weight_quant): weight_quantize_fn()
)
VGG16 4-bit test accuracy: 91.23%


## Target Layer
`model.features[3]` is the 2nd conv layer: 64 -> 64 channels, 3x3 kernel, padding 1, 32x32 feature map.

In [3]:
layer = model_vgg4.features[3]
print(layer)
print('in/out channels:', layer.in_channels, layer.out_channels, '| kernel:', layer.kernel_size, '| padding:', layer.padding)

QuantConv2d(
  64, 64, kernel_size=(3, 3), stride=(1, 1), padding=(1, 1), bias=False
  (weight_quant): weight_quantize_fn()
)
in/out channels: 64 64 | kernel: (3, 3) | padding: (1, 1)


## x_int and w_int
Forward hook on the target layer, one test image.

In [4]:
model_vgg4.eval()
captured = {}

def save_input(module, inputs, output):         # called automatically every time `layer` runs
    captured['x'] = inputs[0].detach().clone()  # inputs is a tuple; inputs[0] is the layer's input tensor

hook = layer.register_forward_hook(save_input)
images, labels = next(iter(testloader))
img = images[0:1].to(device)                    # ONE image, shape [1, 3, 32, 32]
with torch.no_grad():
    model_vgg4(img)
hook.remove()

x = captured['x'][0]                            # drop the batch dim -> [64, 32, 32] (float)
print('hooked input:', x.shape)

hooked input: torch.Size([64, 32, 32])


In [5]:
# w_int: from weight_q (refreshed by the forward pass above), divided by w_delta
w_bit = 4
w_alpha = layer.weight_quant.wgt_alpha.item()
w_delta = w_alpha / (2 ** (w_bit - 1) - 1)
w_int = torch.round(layer.weight_q.detach() / w_delta).cpu()          # [C_out, C_in, k, k]

# x_int: quantize the hooked float input with the layer's own activation quantizer, then divide by x_delta
x_bit = 4
x_alpha = layer.act_alpha.detach()
x_delta = x_alpha.item() / (2 ** x_bit - 1)
x_int = torch.round(act_quantization(x_bit)(x, x_alpha) / x_delta).cpu()   # [C_in, H, W]

print('w_int:', tuple(w_int.shape), 'range', w_int.min().item(), w_int.max().item())
print('x_int:', tuple(x_int.shape), 'range', x_int.min().item(), x_int.max().item())

w_int: (64, 64, 3, 3) range -7.0 7.0
x_int: (64, 32, 32) range 0.0 15.0


## Tiling
The 16x16 array holds one 16 (ic) x 16 (oc) weight block for one kernel pixel. 64/16 = 4 ic tiles x 4 oc tiles = 16 tiles. In tile (ic_tile, oc_tile), PE (row r, col c) holds `w_int[16*oc_tile + c, 16*ic_tile + r, ky, kx]`.

In [6]:
array_size = 16
n_ic_tiles = x_int.shape[0] // array_size      # 4
n_oc_tiles = w_int.shape[0] // array_size      # 4
print('tiles:', n_ic_tiles, 'x', n_oc_tiles, '=', n_ic_tiles * n_oc_tiles)

tiles: 4 x 4 = 16


## Padded, flattened input tiles `a_tile` [ic_tile, oc_tile, ic index (row #), nij]

In [7]:
import torch.nn.functional as F

x_pad = F.pad(x_int, (1, 1, 1, 1))                       # zero-pad H and W (integer domain): [64, 34, 34]
x_flat = x_pad.reshape(x_pad.shape[0], -1)               # nij = y * 34 + x: [64, 1156]
nij_len = x_flat.shape[1]

a_tile = x_flat.reshape(n_ic_tiles, array_size, nij_len)           # [ic_tile, row, nij]
a_tile = a_tile.unsqueeze(1).repeat(1, n_oc_tiles, 1, 1)           # same data re-streamed for every oc tile
print('a_tile:', tuple(a_tile.shape))

a_tile: (4, 4, 16, 1156)


## Step 6: psum [ic_tile, oc_tile, oc index (col #), nij, kij]

In [8]:
kij_len = 9                                   # 3x3 kernel pixels, kij = ky * 3 + kx
psum = torch.zeros(n_ic_tiles, n_oc_tiles, array_size, nij_len, kij_len)

for ic_tile in range(n_ic_tiles):
    for oc_tile in range(n_oc_tiles):
        for kij in range(kij_len):
            ky, kx = kij // 3, kij % 3
            # 16x16 weight block held by the PEs: [oc (col), ic (row)] for this kernel pixel
            w_block = w_int[16*oc_tile : 16*oc_tile + 16, 16*ic_tile : 16*ic_tile + 16, ky, kx]
            # at every time step: weight block x the 16 activations entering the rows -> 16 psums
            psum[ic_tile, oc_tile, :, :, kij] = w_block @ a_tile[ic_tile, oc_tile]   # [16,16] @ [16,1156]

print('psum:', tuple(psum.shape))

psum: (4, 4, 16, 1156, 9)


## Step 7: Accumulate into out [oc index, o_nij]

In [9]:
o_ni_dim = 32                                 # output is 32 x 32
pad_ni_dim = 34                               # padded input is 34 x 34
out = torch.zeros(w_int.shape[0], o_ni_dim * o_ni_dim)       # [oc index, o_nij]

for oy in range(o_ni_dim):
    for ox in range(o_ni_dim):
        o_nij = oy * o_ni_dim + ox
        for kij in range(kij_len):
            ky, kx = kij // 3, kij % 3
            nij = (oy + ky) * pad_ni_dim + (ox + kx)          # padded input pixel that kernel pixel kij sees
            for ic_tile in range(n_ic_tiles):                  # sum over all ic tiles
                for oc_tile in range(n_oc_tiles):              # each oc tile fills its own 16 rows of out
                    out[16*oc_tile : 16*oc_tile + 16, o_nij] += psum[ic_tile, oc_tile, :, nij, kij]

print('out:', tuple(out.shape))

out: (64, 1024)


## Step 8: Check against the integer reference convolution

In [10]:
conv_ref = nn.Conv2d(64, 64, kernel_size=3, stride=1, padding=1, bias=False)
conv_ref.weight = nn.Parameter(w_int)
with torch.no_grad():
    out_ref = conv_ref(x_int.unsqueeze(0))[0].reshape(64, -1)       # [64, 1024]

diff = (out - out_ref).abs()
print('a_tile:', tuple(a_tile.shape))
print('psum  :', tuple(psum.shape))
print('out   :', tuple(out.shape))
print('abs difference: sum = %.1f, max = %.1f' % (diff.sum().item(), diff.max().item()))

a_tile: (4, 4, 16, 1156)
psum  : (4, 4, 16, 1156, 9)
out   : (64, 1024)
abs difference: sum = 0.0, max = 0.0
